# Dr.Egeria Python API

This notebook demonstrates how to call Dr.Egeria programmatically from Python using the two entry points
provided by `md_processing.dr_egeria`:

| Function | When to use |
|---|---|
| `process_md_file()` | Simple scripts — connection parameters passed directly, no client setup needed |
| `process_md_file_v2()` | Reuse an existing `EgeriaTech` client, or call with `await` inside an async context |

Both functions share the same **directive** model:

- **`display`** — parse the file and show what commands were found; no Egeria connection needed
- **`validate`** — connect to Egeria and check that referenced elements exist; no changes are made
- **`process`** — execute the commands and make permanent changes in Egeria; writes a dated receipt file to the Outbox

Connection details and folder paths are loaded from the pyegeria configuration module rather than
being hard-coded, so the same notebook works in both local development and the Egeria Workspaces
container by pointing `PYEGERIA_CONFIG_FILE` at the right JSON file.

---
> **Prerequisites**
> - `pyegeria` and `md_processing` installed (they ship together in the `pyegeria` package)
> - An Egeria view-server reachable at the URL in your config
> - A Dr.Egeria Markdown file to process (placed in the path set by `Dr.Egeria Inbox` in config)

---
## 1. Configuration

pyegeria uses a layered configuration system:

| Priority | Source |
|---|---|
| 1 (lowest) | Built-in Pydantic defaults |
| 2 | JSON config file (`config.json` or `config_workspaces.json`) |
| 3 (highest) | Environment variables (including a `.env` file) |

The JSON file to load is selected by two environment variables:

- `PYEGERIA_CONFIG_DIRECTORY` — directory containing the JSON files (e.g. `…/egeria-python/config`)
- `PYEGERIA_CONFIG_FILE` — filename to load; defaults to `config.json`

### Which config file to use

| Environment | Config file | Notes |
|---|---|---|
| Local dev / PyCharm (egeria-python repo) | `config.json` | Uses `localhost:9443`, `sample-data` root |
| Egeria Workspaces (JupyterLab container) | `config_workspaces.json` | Uses `host.docker.internal:9443`, `/home/jovyan` root |

### 1a. Select your environment

Set `ENVIRONMENT` to `"local"` (PyCharm / egeria-python repo) or `"workspaces"` (Egeria Workspaces
JupyterLab container).  All subsequent cells read from the loaded configuration — no further
changes are needed.

If your config directory is not the `config` folder next to this notebook's repo root, set
`CONFIG_DIR` explicitly; otherwise leave it as `""` and the loader will use whatever
`PYEGERIA_CONFIG_DIRECTORY` is already set to in your environment.

In [1]:
import os

# ── Choose one ────────────────────────────────────────────────────────────────
ENVIRONMENT = "local"        # "local"  → config.json  (PyCharm / egeria-python repo)
# ENVIRONMENT = "workspaces" # "workspaces" → config_workspaces.json  (Egeria Workspaces)
# ─────────────────────────────────────────────────────────────────────────────

# Optional: set the config directory explicitly if it is not already in your environment.
# Leave as "" to rely on PYEGERIA_CONFIG_DIRECTORY already being exported.
CONFIG_DIR = ""   # e.g. "/Users/you/localGit/egeria-python/config" or "/home/jovyan/work/egeria-python/config"

if CONFIG_DIR:
    os.environ["PYEGERIA_CONFIG_DIRECTORY"] = CONFIG_DIR

if ENVIRONMENT == "workspaces":
    os.environ["PYEGERIA_CONFIG_FILE"] = "config_workspaces.json"
else:
    os.environ["PYEGERIA_CONFIG_FILE"] = "config.json"

print(f"Environment : {ENVIRONMENT}")
print(f"Config file : {os.environ.get('PYEGERIA_CONFIG_FILE')}")
print(f"Config dir  : {os.environ.get('PYEGERIA_CONFIG_DIRECTORY', '(from env)')}")

Environment : local
Config file : config.json
Config dir  : (from env)


### 1b. Load and inspect configuration

`load_app_config()` reads the JSON file and overlays any environment variables.  Call it **once**
near the top of your script or notebook.  After that, use `settings.Environment` (and the other
sections) to read values anywhere.

In [2]:
# Suppress noisy SSL warnings when using self-signed certificates in a local lab
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['CURL_CA_BUNDLE'] = ''

from pyegeria.core.config import load_app_config, settings, pretty_print_config

load_app_config()
env = settings.Environment

print(f"Platform URL      : {env.egeria_platform_url}")
print(f"View server       : {env.egeria_view_server}")
print(f"View server URL   : {env.egeria_view_server_url}")
print(f"Dr.Egeria Inbox   : {env.dr_egeria_inbox}")
print(f"Dr.Egeria Outbox  : {env.dr_egeria_outbox}")
print(f"Pyegeria Root     : {env.pyegeria_root}")
print(f"Jupyter mode      : {env.egeria_jupyter}")

Platform URL      : https://localhost:9443
View server       : qs-view-server
View server URL   : https://localhost:9443
Dr.Egeria Inbox   : egeria-inbox/dr-egeria-inbox
Dr.Egeria Outbox  : egeria-outbox/dr-egeria-outbox
Pyegeria Root     : /Users/dwolfson/localGit/egeria-python/examples/Jupyter Notebooks
Jupyter mode      : True


In [ ]:
# Optional: pretty-print the full effective configuration (passwords masked)
pretty_print_config(safe=True)

### 1c. Credentials

User credentials are intentionally **not** stored in the JSON config files.  Supply them via
environment variables (`EGERIA_USER`, `EGERIA_USER_PASSWORD`), a `.env` file, or override the
defaults in this cell for interactive use.

In [ ]:
EGERIA_USER     = os.environ.get("EGERIA_USER",          "erinoverview")
EGERIA_PASSWORD = os.environ.get("EGERIA_USER_PASSWORD", "secret")

print(f"User : {EGERIA_USER}")

### 1d. Jupyter async compatibility

Jupyter kernels run their own event loop, so `asyncio.run()` raises a `RuntimeError` inside a cell.
Two options:

1. **Use `await` directly** (shown in Part B) — works without extra packages.
2. **Install `nest_asyncio`** — allows `asyncio.run()` to work normally everywhere (useful when
   copy-pasting script code into a notebook).

The cell below applies `nest_asyncio` if it is available.

In [ ]:
try:
    import nest_asyncio
    nest_asyncio.apply()
    print("nest_asyncio applied — asyncio.run() will work inside notebook cells.")
except ImportError:
    print("nest_asyncio not installed. Use 'await' for async calls (see Part B).")
    print("Install with: pip install nest_asyncio")

### 1e. Choose your input file

Set `INPUT_FILE` to the name of a Dr.Egeria Markdown file.  You can use:

- A bare filename — Dr.Egeria looks for it inside the `Dr.Egeria Inbox` path from config.
- An absolute path — used as-is.

In [ ]:
INPUT_FILE = "dr_egeria_intro_part1.md"   # filename inside Dr.Egeria Inbox, or an absolute path

---
## Part A — `process_md_file()` (synchronous)

`process_md_file` is the simplest entry point. You pass connection parameters directly and it
handles creating the `EgeriaTech` client internally.  Here we read those parameters from
`settings.Environment` so nothing is hard-coded.

```
process_md_file(
    input_file,      # .md file path — absolute, or relative to Dr.Egeria Inbox
    output_folder,   # subfolder inside Dr.Egeria Outbox for the receipt; "" for root
    directive,       # "display" | "validate" | "process"
    server,          # view-server name          ← env.egeria_view_server
    url,             # platform URL              ← env.egeria_view_server_url
    userid,          # user id
    user_pass,       # password
    parse_summary,   # "all" | "errors" | "none"  (default "none")
    attribute_logs,  # "debug" | "info" | "none"  (default "debug")
    usage_level,     # "Basic" | "Advanced"       (default None → "Basic")
    summary_only,    # True → show only the summary table (default False)
    debug,           # True → log every Egeria API request (default False)
)
```

### A1. Display — inspect commands without connecting to Egeria

With `directive="display"` no Egeria connection is made — useful for a quick sanity-check of the file.

In [ ]:
from md_processing.dr_egeria import process_md_file

process_md_file(
    input_file=INPUT_FILE,
    output_folder="",
    directive="display",
    server=env.egeria_view_server,
    url=env.egeria_view_server_url,
    userid=EGERIA_USER,
    user_pass=EGERIA_PASSWORD,
)

### A2. Validate — check commands against a live Egeria instance

`directive="validate"` connects to Egeria to verify that every referenced element exists and that
the command attributes are correct.  No changes are made.

In [ ]:
from md_processing.dr_egeria import process_md_file

process_md_file(
    input_file=INPUT_FILE,
    output_folder="",
    directive="validate",
    server=env.egeria_view_server,
    url=env.egeria_view_server_url,
    userid=EGERIA_USER,
    user_pass=EGERIA_PASSWORD,
    parse_summary="errors",   # print per-command summaries only when there are errors
)

### A3. Process — execute commands and write a receipt file

`directive="process"` executes every command in the file and writes a dated receipt document to the
Outbox path defined in config (`Dr.Egeria Outbox`).

> **This makes permanent changes in Egeria.** Run `validate` first to confirm the file is correct.

In [ ]:
from md_processing.dr_egeria import process_md_file

process_md_file(
    input_file=INPUT_FILE,
    output_folder="",          # receipt goes to root of Dr.Egeria Outbox
    directive="process",
    server=env.egeria_view_server,
    url=env.egeria_view_server_url,
    userid=EGERIA_USER,
    user_pass=EGERIA_PASSWORD,
    summary_only=True,         # show only the summary table — reduces notebook output
)

---
## Part B — `process_md_file_v2()` (async / `await`)

`process_md_file_v2` is the async engine underneath `process_md_file`. Use it when you want to:

- Reuse a single `EgeriaTech` client across multiple calls
- Call with `await` inside an existing async context
- Have fine-grained control over client creation

```
await process_md_file_v2(
    input_file,      # .md file path
    output_folder,   # subfolder inside Dr.Egeria Outbox; "" for root
    directive,       # "display" | "validate" | "process"
    client,          # pre-built EgeriaTech instance
    parse_summary,   # "all" | "errors" | "none"  (default "none")
    attribute_logs,  # "debug" | "info" | "none"  (default "info")
    usage_level,     # "Basic" | "Advanced"       (default None → "Basic")
    summary_only,    # True → show only the summary table (default False)
    debug,           # True → log every Egeria API request (default False)
)
```

### B1. Build a shared client

Create the `EgeriaTech` client once using values from `settings.Environment` and reuse it for all
calls in this notebook.

In [ ]:
from pyegeria import EgeriaTech

client = EgeriaTech(
    server_name=env.egeria_view_server,
    platform_url=env.egeria_view_server_url,
    user_id=EGERIA_USER,
)
client.create_egeria_bearer_token(EGERIA_USER, EGERIA_PASSWORD)
print(f"EgeriaTech client ready — {env.egeria_view_server} @ {env.egeria_view_server_url}")

### B2. Validate using `await`

In [ ]:
from md_processing.dr_egeria import process_md_file_v2

await process_md_file_v2(
    input_file=INPUT_FILE,
    output_folder="",
    directive="validate",
    client=client,
)

### B3. Process using `await`

> **This makes permanent changes in Egeria.**

In [ ]:
from md_processing.dr_egeria import process_md_file_v2

await process_md_file_v2(
    input_file=INPUT_FILE,
    output_folder="",
    directive="process",
    client=client,
    summary_only=True,
)

### B4. Process multiple files with the same client

Reusing a single client avoids repeating authentication for every file.

In [ ]:
from md_processing.dr_egeria import process_md_file_v2

files_to_process = [
    "glossary.md",
    "projects.md",
    "data_structures.md",
]

for md_file in files_to_process:
    print(f"\n--- Processing {md_file} ---")
    await process_md_file_v2(
        input_file=md_file,
        output_folder="batch-run",
        directive="process",
        client=client,
        summary_only=True,
    )

### B5. Advanced options — usage level and debug mode

- `usage_level="Advanced"` exposes additional optional attributes during validation.
  The default is read from `User Profile → Egeria Usage Level` in config.
- `debug=True` prints every Egeria API request URL and body — useful for troubleshooting.

In [ ]:
from md_processing.dr_egeria import process_md_file_v2

# Read usage level from config (falls back to "Basic" if not set)
usage_level = settings.User_Profile.egeria_usage_level or "Basic"

await process_md_file_v2(
    input_file=INPUT_FILE,
    output_folder="",
    directive="validate",
    client=client,
    usage_level=usage_level,
    parse_summary="all",
    debug=False,            # set True to see every API call
)

---
## Part C — Using `asyncio.run()` (requires `nest_asyncio`)

If `nest_asyncio` was applied in the setup cell above, you can call `asyncio.run()` exactly as you
would in a plain Python script.  This is handy for copy-pasting script code into a notebook without
changing `asyncio.run(...)` to `await ...`.

The pattern below is identical to what you would write in a standalone `.py` file.

In [ ]:
import asyncio
from pyegeria import EgeriaTech
from pyegeria.core.config import load_app_config, settings
from md_processing.dr_egeria import process_md_file_v2

# load_app_config() is idempotent — safe to call again; returns the cached config
load_app_config()
env = settings.Environment

script_client = EgeriaTech(
    server_name=env.egeria_view_server,
    platform_url=env.egeria_view_server_url,
    user_id=EGERIA_USER,
)
script_client.create_egeria_bearer_token(EGERIA_USER, EGERIA_PASSWORD)

# asyncio.run() works here because nest_asyncio.apply() was called at startup
asyncio.run(
    process_md_file_v2(
        input_file=INPUT_FILE,
        output_folder="",
        directive="validate",
        client=script_client,
        summary_only=True,
    )
)

---
## Appendix — Environment variable reference

These variables override the corresponding JSON config values.
Set them in your shell, a `.env` file, or `os.environ` before calling `load_app_config()`.

| Variable | JSON key | Purpose |
|---|---|---|
| `PYEGERIA_CONFIG_DIRECTORY` | `Pyegeria Config Directory` | Directory containing the JSON config file |
| `PYEGERIA_CONFIG_FILE` | `Egeria Config File` | JSON filename (`config.json` / `config_workspaces.json`) |
| `PYEGERIA_ROOT_PATH` | `Pyegeria Root` | Base path for inbox/outbox/mermaid folders |
| `EGERIA_PLATFORM_URL` | `Egeria Platform URL` | Platform URL |
| `EGERIA_VIEW_SERVER` | `Egeria View Server` | View-server name |
| `EGERIA_VIEW_SERVER_URL` | `Egeria View Server URL` | View-server URL |
| `DR_EGERIA_INBOX_PATH` | `Dr.Egeria Inbox` | Inbox folder path |
| `DR_EGERIA_OUTBOX_PATH` | `Dr.Egeria Outbox` | Outbox folder path |
| `EGERIA_USER` | `user_name` in User Profile | Egeria user id |
| `EGERIA_USER_PASSWORD` | `user_pwd` in User Profile | Egeria password |
| `EGERIA_USAGE_LEVEL` | `Egeria Usage Level` | `Basic` or `Advanced` |
| `EGERIA_JUPYTER` | `Egeria Jupyter` | `True` for notebook-friendly Rich output |
| `PYEGERIA_NORMALIZE_MERMAID` | `Egeria Normalize Mermaid` | `True` to normalise Mermaid syntax |